# FastAPI: From Basics to Intermediate (with Trading-Flavored Examples)

This notebook is a **from-scratch to intermediate** guide to **FastAPI**, a modern,
async-friendly web framework for building APIs with Python.

We’ll use simple **trading-flavored** examples (symbols, orders, prices) to make
endpoints realistic without needing a real exchange or database.

### Contents

- [1. What is FastAPI and Why Use It?](#1-what-is-fastapi-and-why-use-it)
- [2. Minimal App: Hello World and Simple Endpoint](#2-minimal-app-hello-world-and-simple-endpoint)
- [3. Path, Query, and Body Parameters](#3-path-query-and-body-parameters)
- [4. Pydantic Models for Requests and Responses](#4-pydantic-models-for-requests-and-responses)
- [5. Dependency Injection (DB/Client, Settings, Auth)](#5-dependency-injection-dbclient-settings-auth)
- [6. Background Tasks and Scheduled Work](#6-background-tasks-and-scheduled-work)
- [7. Error Handling, Validation, and Responses](#7-error-handling-validation-and-responses)
- [8. Testing FastAPI Apps (TestClient)](#8-testing-fastapi-apps-testclient)
- [9. Async I/O and Combining with External Services](#9-async-io-and-combining-with-external-services)
- [10. Best Practices and Project Structure](#10-best-practices-and-project-structure)
- [11. Running Your FastAPI App with Uvicorn](#11-running-your-fastapi-app-with-uvicorn)

## 1. What is FastAPI and Why Use It?

FastAPI is a **high-performance, async-first** web framework for building APIs in Python.
Key properties:

- Built on **Starlette** (ASGI) and **Pydantic**.
- First-class support for **async def** endpoints.
- Automatic **OpenAPI/Swagger** docs and interactive UI.
- Strong **type hints** → better editor support and runtime validation.

For trading/quant systems, FastAPI is a good fit for:

- **REST gateways**: order submission, positions, PnL, risk metrics.
- **Monitoring/observability endpoints**.
- **Service-to-service APIs** within your trading platform.

We’ll mostly focus on simple in-memory examples so you can run everything inside
this notebook or as a single `main.py` script.

In [ ]:
# 1.1 Check FastAPI and Uvicorn availability

try:
    import fastapi
    from fastapi import FastAPI
    print("FastAPI version:", fastapi.__version__)
except ImportError:
    print("Install FastAPI: pip install fastapi[all]")

try:
    import uvicorn
    print("Uvicorn version:", uvicorn.__version__)
except ImportError:
    print("Install Uvicorn: pip install uvicorn[standard]")

FastAPI version: 0.115.12
Install Uvicorn: pip install uvicorn[standard]


## 2. Minimal App: Hello World and Simple Endpoint

A minimal FastAPI app lives in a `main.py` like:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "Hello World"}
```

Run it with:

```bash
uvicorn main:app --reload
```

In this notebook, we’ll define `app` in-code and use FastAPI’s `TestClient` to
exercise endpoints without running a real server process (cleaner for examples).
You can copy/paste the code into a `main.py` for a real service.

In [ ]:
# 2.1 Minimal FastAPI app + TestClient demo

from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Trading API Demo")


@app.get("/")
async def root():
    return {"message": "Hello from Trading API"}


client = TestClient(app)
response = client.get("/")
print("Status:", response.status_code)
print("JSON:", response.json())

Status: 200
JSON: {'message': 'Hello from Trading API'}


## 3. Path, Query, and Body Parameters

FastAPI automatically parses and validates **path**, **query**, and **body** parameters
based on type hints.

We’ll build endpoints like:

- `GET /symbols/{symbol}` → path param `symbol: str`.
- `GET /prices` → query params `symbol: str`, optional `limit: int = 100`.
- `POST /orders` → body model with order details.

This is the core of how you design REST APIs with FastAPI.

In [ ]:
# 3.1 Path and query parameters

from typing import Optional
from fastapi import Query

FAKE_SYMBOLS = ["AAPL", "MSFT", "GOOG", "TSLA"]
FAKE_PRICES = {"AAPL": 180.0, "MSFT": 320.0, "GOOG": 140.0, "TSLA": 250.0}


@app.get("/symbols")
async def list_symbols(limit: int = Query(10, ge=1, le=100)):
    """List available symbols, capped by `limit` query param."""
    return FAKE_SYMBOLS[:limit]


@app.get("/symbols/{symbol}")
async def get_symbol(symbol: str):
    """Return info about a single symbol (from path)."""
    if symbol not in FAKE_SYMBOLS:
        return {"symbol": symbol, "available": False}
    return {"symbol": symbol, "available": True}


@app.get("/prices")
async def get_price(symbol: str, with_timestamp: Optional[bool] = False):
    """Get the latest price for a symbol; `with_timestamp` is a query flag."""
    price = FAKE_PRICES.get(symbol)
    if price is None:
        return {"symbol": symbol, "error": "unknown symbol"}
    result = {"symbol": symbol, "price": price}
    if with_timestamp:
        import time as _time
        result["ts"] = _time.time()
    return result


print("GET /symbols:", client.get("/symbols").json())
print("GET /symbols/AAPL:", client.get("/symbols/AAPL").json())
print("GET /prices?symbol=AAPL&with_timestamp=true:", client.get("/prices", params={"symbol": "AAPL", "with_timestamp": True}).json())

GET /symbols: ['AAPL', 'MSFT', 'GOOG', 'TSLA']
GET /symbols/AAPL: {'symbol': 'AAPL', 'available': True}
GET /prices?symbol=AAPL&with_timestamp=true: {'symbol': 'AAPL', 'price': 180.0, 'ts': 1775513212.1983578}


## 4. Pydantic Models for Requests and Responses

FastAPI uses **Pydantic** (v1 or v2) for data models:

- Define request/response schemas as Python classes with type hints.
- Get **validation**, **docs**, and **serialization** for free.

We’ll define:

- `OrderIn` – request body model for creating an order.
- `OrderOut` – response model with server-generated fields (e.g. order ID, status).

This is the recommended way to structure non-trivial JSON payloads.

In [ ]:
# 4.1 Pydantic order models and in-memory store

from pydantic import BaseModel, Field
from enum import Enum

class Side(str, Enum):
    buy = "BUY"
    sell = "SELL"


class OrderType(str, Enum):
    market = "MARKET"
    limit = "LIMIT"


class OrderIn(BaseModel):
    symbol: str = Field(..., example="AAPL")
    side: Side
    type: OrderType
    quantity: float = Field(..., gt=0)
    limit_price: float | None = Field(None, description="Required for LIMIT orders")


class OrderOut(BaseModel):
    id: int
    symbol: str
    side: Side
    type: OrderType
    quantity: float
    limit_price: float | None
    status: str


ORDERS: dict[int, OrderOut] = {}
_next_order_id = 1


@app.post("/orders", response_model=OrderOut, status_code=201)
async def create_order(order: OrderIn):
    """Create a new order (in-memory).

    In a real app, this would call a broker or matching engine.
    """
    global _next_order_id

    if order.type is OrderType.limit and order.limit_price is None:
        # Basic validation beyond type hints
        return {"detail": "limit_price required for LIMIT orders"}

    oid = _next_order_id
    _next_order_id += 1
    order_out = OrderOut(
        id=oid,
        symbol=order.symbol,
        side=order.side,
        type=order.type,
        quantity=order.quantity,
        limit_price=order.limit_price,
        status="accepted",
    )
    ORDERS[oid] = order_out
    return order_out


@app.get("/orders/{order_id}", response_model=OrderOut)
async def get_order(order_id: int):
    order = ORDERS.get(order_id)
    if not order:
        from fastapi import HTTPException

        raise HTTPException(status_code=404, detail="Order not found")
    return order


# Quick round-trip with TestClient
resp = client.post(
    "/orders",
    json={
        "symbol": "AAPL",
        "side": "BUY",
        "type": "LIMIT",
        "quantity": 100,
        "limit_price": 180.0,
    },
)
print("POST /orders status:", resp.status_code)
created = resp.json()
print("Created order:", created)
print("GET /orders/{id}:", client.get(f"/orders/{created['id']}").json())

POST /orders status: 201
Created order: {'id': 1, 'symbol': 'AAPL', 'side': 'BUY', 'type': 'LIMIT', 'quantity': 100.0, 'limit_price': 180.0, 'status': 'accepted'}
GET /orders/{id}: {'id': 1, 'symbol': 'AAPL', 'side': 'BUY', 'type': 'LIMIT', 'quantity': 100.0, 'limit_price': 180.0, 'status': 'accepted'}


## 5. Dependency Injection (DB/Client, Settings, Auth)

FastAPI has a simple but powerful **dependency injection** system via `Depends`.
You declare function parameters with `Depends(...)`, and FastAPI resolves them
per-request. This is ideal for:

- Database sessions
- HTTP clients
- Settings / configuration
- Simple auth or permission checks

FastAPI's `Depends` system helps you automatically inject "dependencies" (shared objects or logic)
into your endpoint functions, like database connections or config settings, without having to
create or pass them manually each time. This keeps your code clean and avoids repeating yourself.

If you don't use `Depends`, you'd have to write extra code to create, pass, or clean up these
objects in every endpoint, leading to more errors and harder maintenance. `Depends` makes
it easy to share common resources, handle setup and teardown, and write reusable, testable code.

We’ll simulate a **settings object** and a **fake database client** and inject them into an endpoint. Notice that the usage of `Depends` makes it possible to avoid calling or manually instantiating `get_db` every time at the beginning of the `get_positions` endpoint. Being already included in the function's scope.

In [ ]:
# 5.1 Dependencies: settings and fake DB client

from fastapi import Depends


class Settings(BaseModel):
    app_name: str = "Trading API Demo"
    risk_limit: float = 1_000_000.0


def get_settings() -> Settings:
    # In real apps, you might read from env vars or config files
    return Settings()


class FakeDB:
    def __init__(self):
        self._positions: dict[str, float] = {"AAPL": 100.0, "MSFT": -50.0}

    def get_position(self, symbol: str) -> float | None:
        return self._positions.get(symbol)


db_instance = FakeDB()


def get_db() -> FakeDB:
    # Could open/close connections per request; here we reuse a singleton
    return db_instance


@app.get("/positions/{symbol}")
async def get_position(
    symbol: str,
    db: FakeDB = Depends(get_db),
    settings: Settings = Depends(get_settings),
):
    qty = db.get_position(symbol)
    return {
        "symbol": symbol,
        "quantity": qty,
        "risk_limit": settings.risk_limit,
        "within_limit": (abs(qty or 0) * 100) < settings.risk_limit,
    }


print("GET /positions/AAPL:", client.get("/positions/AAPL").json())

GET /positions/AAPL: {'symbol': 'AAPL', 'quantity': 100.0, 'risk_limit': 1000000.0, 'within_limit': True}


## 6. Background Tasks and Scheduled Work

Sometimes you want to **acknowledge a request quickly** but do extra work in the
background (e.g. logging, notifying another system, slow persistence).

FastAPI provides **`BackgroundTasks`** for simple fire-and-forget work that runs
*after* the response is sent.

We’ll extend our order endpoint to queue a background task that "logs" the order
(in reality, appends to a list with a timestamp).

In [ ]:
# 6.1 Using BackgroundTasks

from fastapi import BackgroundTasks
import time as _time

ORDER_LOG: list[dict] = []


def log_order(order: OrderOut) -> None:
    # This runs after the response has been sent to the client
    ORDER_LOG.append({"ts": _time.time(), "order": order.dict()})


@app.post("/orders_with_bg", response_model=OrderOut, status_code=201)
async def create_order_with_bg(order: OrderIn, background_tasks: BackgroundTasks):
    global _next_order_id
    if order.type is OrderType.limit and order.limit_price is None:
        from fastapi import HTTPException

        raise HTTPException(status_code=400, detail="limit_price required for LIMIT orders")

    oid = _next_order_id
    _next_order_id += 1
    order_out = OrderOut(
        id=oid,
        symbol=order.symbol,
        side=order.side,
        type=order.type,
        quantity=order.quantity,
        limit_price=order.limit_price,
        status="accepted",
    )
    ORDERS[oid] = order_out

    background_tasks.add_task(log_order, order_out)
    return order_out


resp = client.post(
    "/orders_with_bg",
    json={
        "symbol": "MSFT",
        "side": "SELL",
        "type": "MARKET",
        "quantity": 10,
    },
)
print("Created order_with_bg:", resp.json())
print("Order log size (may be 0 if task not run yet):", len(ORDER_LOG))

Created order_with_bg: {'id': 2, 'symbol': 'MSFT', 'side': 'SELL', 'type': 'MARKET', 'quantity': 10.0, 'limit_price': None, 'status': 'accepted'}
Order log size (may be 0 if task not run yet): 1


C:\Users\GSL\AppData\Local\Temp\ipykernel_37456\1840248732.py:11: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  ORDER_LOG.append({"ts": _time.time(), "order": order.dict()})


## 7. Error Handling, Validation, and Responses

FastAPI integrates tightly with Pydantic and HTTP semantics:

- Raise `HTTPException` with `status_code` and `detail` for controlled errors.
- Use `response_model` to shape output and hide internal fields.
- Add extra validation in your endpoint or via Pydantic validators.

We already used `HTTPException` in `get_order` and `create_order_with_bg`. Here we add
an endpoint that demonstrates **custom error responses** and a strongly-typed
`response_model` with **optional fields**.

In [ ]:
# 7.1 Custom error response and response_model

from fastapi import HTTPException


class PositionResponse(BaseModel):
    symbol: str
    quantity: float | None
    avg_price: float | None
    error: str | None = None


@app.get("/positions_detailed/{symbol}", response_model=PositionResponse)
async def get_position_detailed(symbol: str, db: FakeDB = Depends(get_db)):
    qty = db.get_position(symbol)
    if qty is None:
        # You can raise HTTPException or return an error object matching response_model
        raise HTTPException(status_code=404, detail=f"no position for {symbol}")
    # In this toy example we don't track avg_price
    return PositionResponse(symbol=symbol, quantity=qty, avg_price=None)


print("GET /positions_detailed/MSFT:", client.get("/positions_detailed/MSFT").json())
print("GET /positions_detailed/UNK status:", client.get("/positions_detailed/UNK").status_code)

GET /positions_detailed/MSFT: {'symbol': 'MSFT', 'quantity': -50.0, 'avg_price': None, 'error': None}
GET /positions_detailed/UNK status: 404


## 8. Testing FastAPI Apps (TestClient)

We’ve already used **`TestClient`** (built on `requests`) to call endpoints directly
from Python. This is how you write **unit/integration tests** without running an
actual web server process.

Key ideas:

- Import your `app`.
- Create `TestClient(app)`.
- Use `.get()`, `.post()`, etc. just like with `requests`.

We’ll write a few small tests that exercise the order and position endpoints.

In [ ]:
# 8.1 Example tests (can be moved to test_*.py)

from fastapi.testclient import TestClient

client = TestClient(app)


def test_create_and_get_order():
    payload = {
        "symbol": "AAPL",
        "side": "BUY",
        "type": "MARKET",
        "quantity": 5,
    }
    r = client.post("/orders", json=payload)
    assert r.status_code == 201
    data = r.json()
    oid = data["id"]
    r2 = client.get(f"/orders/{oid}")
    assert r2.status_code == 200
    data2 = r2.json()
    assert data2["symbol"] == "AAPL"


def test_get_unknown_symbol_price():
    r = client.get("/prices", params={"symbol": "FOO"})
    assert r.status_code == 200
    assert r.json()["error"] == "unknown symbol"


# Run tests inline (for demo only)
if __name__ == "__main__":
    test_create_and_get_order()
    test_get_unknown_symbol_price()
    print("Inline tests passed")
else:
    # In notebook, just call them directly
    test_create_and_get_order()
    test_get_unknown_symbol_price()
    print("Inline tests passed (notebook)")

Inline tests passed


## 9. Async I/O and Combining with External Services

FastAPI endpoints can be **async**, so you can safely call other async libraries inside
(e.g. `httpx.AsyncClient`, `aiohttp`, async DB drivers). This is how you build a
**gateway** that fans out to other services.

We’ll define an endpoint that calls a **public crypto REST API** (Binance) using
`httpx` (an async HTTP client), then returns a combined response. In production
code, you might use `aiohttp` as well; FastAPI doesn’t care which async client you use.

### Difference between "httpx" and "aiohttp"

- **httpx** and **aiohttp** are both Python HTTP client libraries that support async operations.

- **httpx**:
    - Modern, fully-featured HTTP client from the authors of Requests.
    - Has a Requests-like API and supports both synchronous and asynchronous usage.
    - Built-in HTTP/2, connection pooling, browser-style cookies, and more.
    - Pure client: only provides HTTP client functionality (not a web server).
    - Syntax example: `async with httpx.AsyncClient() as client: ...`

- **aiohttp**:
    - One of the original async HTTP libraries for Python.
    - Supports both HTTP client *and* server functionality.
    - Often used in older async Python projects and for building small web servers.
    - Client API is different from Requests but well-documented.
    - Syntax example: `async with aiohttp.ClientSession() as session: ...`

- **Which to use?**
    - For most new async client-only code, `httpx` is preferred for its modern API and compatibility with Requests.
    - If you need a lightweight async web server or are maintaining legacy code, `aiohttp` may be appropriate.

In [11]:
# 9.1 Async endpoint that calls a public crypto API

import httpx

BINANCE_BASE = "https://api.binance.com"


async def fetch_binance_price(symbol: str) -> float | None:
    url = f"{BINANCE_BASE}/api/v3/ticker/price"
    params = {"symbol": symbol}
    try:
        async with httpx.AsyncClient(timeout=5.0) as client_httpx:
            resp = await client_httpx.get(url, params=params)
            resp.raise_for_status()
            data = resp.json()
            return float(data["price"])
    except Exception as e:
        print(f"Error fetching {symbol} from Binance:", e)
        return None


@app.get("/external/price/{symbol}")
async def external_price(symbol: str):
    price = await fetch_binance_price(symbol)
    if price is None:
        raise HTTPException(status_code=502, detail="upstream price service failed")
    return {"symbol": symbol, "price": price}


print("GET /external/price/BTCUSDT (may hit real API):")
print(client.get("/external/price/BTCUSDT").json())

GET /external/price/BTCUSDT (may hit real API):
{'symbol': 'BTCUSDT', 'price': 68602.2}


## 10. Best Practices and Project Structure

**Project structure** (small service):

```text
app/
  __init__.py
  main.py          # FastAPI app, routes include
  models.py        # Pydantic models
  routers/
    orders.py      # /orders endpoints
    positions.py   # /positions endpoints
  deps.py          # dependency providers (DB, settings, auth)
  config.py        # settings and environment loading

tests/
  test_orders.py
  test_positions.py
```

**Operational tips**

- Use **Uvicorn** or **Gunicorn+Uvicorn workers** to serve the app in production (see [section 11](#11-running-your-fastapi-app-with-uvicorn) for CLI, `uvicorn.run`, and worker notes).
- Configure **logging** and **access logs**; consider metrics (Prometheus) and tracing.
- Treat Pydantic models as your **API contracts**; version them carefully.
- Validate inputs rigorously; don’t trust client-provided symbols, sizes, or prices.

**When to choose FastAPI**

- You need a **well-typed, async-friendly REST API** to expose trading functionality
  (orders, positions, risk, monitoring).
- You want quick iteration but still care about **performance** and **validation**.

For ultra-low-latency order entry paths, you may prefer off-process gateways (C++
with pybind11, raw TCP/UDP, etc.), but FastAPI is an excellent fit for most
control-plane and monitoring APIs in trading systems.

## 11. Running Your FastAPI App with Uvicorn

Throughout this notebook we exercised the **same** `app` instance with `TestClient`. That is ideal for learning and automated tests. To accept **real HTTP** traffic from browsers, `curl`, or other services, you run an **ASGI server**. **Uvicorn** is the usual choice with FastAPI: it implements ASGI efficiently and works with the **async** endpoints we built (for example `/external/price/{symbol}` using `httpx`).

### 11.1 The `module:app` import string

Uvicorn loads your application from an import path of the form **`module:variable`**:

- `main:app` means: Python imports `app` from `main.py` in the current working directory (or on `PYTHONPATH`).
- If you moved the routes from this notebook into `trading_api.py` with `app = FastAPI(...)`, you would run `uvicorn trading_api:app`.

That `app` is exactly what we passed to `TestClient(app)` in [section 8](#8-testing-fastapi-apps-testclient): same paths (`/orders`, `/prices`, `/positions/{symbol}`, `/docs`, etc.), dependencies, and validation—only the **transport** changes (in-process test client vs TCP HTTP).

### 11.2 CLI: dev server, host, port, and reload

From the directory that contains your module (or with `PYTHONPATH` set), run:

```bash
uvicorn main:app --reload
```

Useful flags for local development:

- **`--reload`** — restart the process when code changes (development only; not for production hardening).
- **`--host 0.0.0.0`** — listen on all interfaces (e.g. hit the API from another device on your LAN).
- **`--port 8000`** — listen port (8000 is the default if omitted).
- **`--log-level debug`** — more verbose request and application logging.

Then open **interactive OpenAPI docs** at `http://127.0.0.1:8000/docs` and try the same trading-style endpoints you called with `client.get` / `client.post` earlier (`/symbols`, `/orders`, `/positions/AAPL`, and so on).

### 11.3 Programmatic startup with `uvicorn.run`

In a small `main.py` (or any entry script), you can start Uvicorn from Python instead of the shell. Two common patterns:

**Pass the app object** (fine for simple scripts):

```python
import uvicorn
from fastapi import FastAPI

app = FastAPI()

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)
```

**Pass the `module:app` string** (recommended if you use **`reload=True`**, so the reloader can re-import your module cleanly):

```python
if __name__ == "__main__":
    uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=True)
```

Avoid calling `uvicorn.run(...)` in the middle of this notebook: it **blocks the kernel** until you interrupt it. For notebooks, keep using `TestClient`; copy the `app` and routes into a `.py` file when you want a long-lived server.

In [ ]:
# 11.3 (notebook) — confirm Uvicorn is available; same stack as section 1.1

try:
    import uvicorn

    print("Uvicorn version:", uvicorn.__version__)
    print("Example (run from a .py file, not here): uvicorn.run(app, host='127.0.0.1', port=8000)")
    print("Or CLI from the folder with your module: uvicorn main:app --reload")
except ImportError:
    print("Install Uvicorn: pip install uvicorn[standard]")

### 11.4 Workers, logging, and production-shaped deployments

- **Single worker** (default) — simplest; matches how we reason about the in-memory `ORDERS` and `FAKE_PRICES` examples in one process.
- **Multiple workers** — `uvicorn main:app --workers 4` runs several processes. Each worker has **its own memory**, so our demo **`ORDERS` dict is not shared** across workers. For a real trading API you would persist orders in a database or message broker and treat the app as stateless.
- **Process managers** — on Linux, **Gunicorn** with `uvicorn.workers.UvicornWorker` is a common way to manage worker processes and integrate with init systems.
- **HTTPS** — often handled by a **reverse proxy** (nginx, Caddy, or a cloud load balancer) in front of Uvicorn, which keeps the Python app on plain HTTP locally.

Together with FastAPI, Uvicorn is how you turn the patterns in this notebook—path/query/body params, Pydantic models, dependencies, background tasks, and async upstream calls—into a **real service** you can deploy and monitor.